# Axis 2: Flux Interaction Affordance Probing

This notebook runs the Axis 2 interaction affordance probing pipeline on Colab Pro.

**Requirements:**
- Colab Pro with A100 GPU (40GB VRAM)
- AGD20K dataset on Google Drive

**Pipeline:**
1. Clone repo and install dependencies
2. Download/link AGD20K dataset
3. Smoke test (script 09)
4. Run interaction probing (script 10)
5. Generate report (script 11)
6. Save results to Drive

## Cell 1: Environment Setup

In [ ]:
# Clone the repo
!git clone https://github.com/aleksantari/VLA-affordance.git
%cd VLA-affordance

# Checkout the feature branch
!git checkout nj-features

# Install base + Axis 2 dependencies
!pip install -e . -q
!pip install -r requirements_axis2.txt -q

print("\n✓ Environment setup complete")

## Cell 2: Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name()
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    
    if vram_gb < 20:
        print("\n⚠ WARNING: Less than 20GB VRAM detected.")
        print("Flux requires ~24GB. You may need to:")
        print("  1. Use Runtime > Change runtime type > A100")
        print("  2. Or enable CPU offloading (slower)")
    else:
        print("\n✓ Sufficient VRAM for Flux")
else:
    print("❌ No GPU detected. Enable GPU in Runtime > Change runtime type")

## Cell 3: Download/Link AGD20K Dataset

**Option A**: If AGD20K is already on your Google Drive, mount and symlink.

**Option B**: Download directly (slower).

Update the `DRIVE_AGD20K_PATH` below to match your Drive location.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Option A: Symlink from Drive ──
# Update this path to where AGD20K is on your Drive
DRIVE_AGD20K_PATH = "/content/drive/MyDrive/datasets/AGD20K"

import os
if os.path.exists(DRIVE_AGD20K_PATH):
    !python data/download_agd20k.py --from_drive {DRIVE_AGD20K_PATH} --output_dir ./data/agd20k
else:
    print(f"Drive path not found: {DRIVE_AGD20K_PATH}")
    print("Falling back to direct download...")
    # ── Option B: Direct download ──
    !python data/download_agd20k.py --output_dir ./data/agd20k

# Validate
!python data/download_agd20k.py --validate_only --output_dir ./data/agd20k

## Cell 4: Smoke Test (Script 09)

Verify Flux loads correctly and attention extraction works.

In [ ]:
# Start with schnell (fast, 4 steps) for verification
!python scripts/09_setup_flux.py --model schnell

In [ ]:
# Display sample visualizations
from IPython.display import Image, display
from pathlib import Path

figures_dir = Path("./results/figures/axis2")
for fig_path in sorted(figures_dir.glob("flux_test_*.png")):
    print(f"\n{fig_path.name}:")
    display(Image(filename=str(fig_path), width=600))

## Cell 5: Run Interaction Probing (Script 10)

This is the main compute cell. On A100:
- Schnell: ~30-60 min for full dataset
- Dev: ~2-4 hours for full dataset

Start with a quick test on a few categories, then run the full evaluation.

In [ ]:
# Quick test: 3 categories, 5 samples each (schnell)
!python scripts/10_run_interaction_probing.py \
    --model schnell \
    --categories cut hold pour \
    --max_per_category 5 \
    --save_attention_maps

In [ ]:
# Full run with schnell (faster iteration)
# Uncomment and run when ready:

# !python scripts/10_run_interaction_probing.py \
#     --model schnell \
#     --save_attention_maps

In [ ]:
# Full run with dev (final results)
# Uncomment and run for publication-quality results:

# !python scripts/10_run_interaction_probing.py \
#     --model dev \
#     --save_attention_maps

## Cell 6: Generate Report (Script 11)

In [ ]:
!python scripts/11_generate_axis2_report.py

In [ ]:
# Display report figures
from IPython.display import Image, display
from pathlib import Path

figures_dir = Path("./results/figures/axis2")
for fig_path in sorted(figures_dir.glob("axis2_*.png")):
    print(f"\n{fig_path.name}:")
    display(Image(filename=str(fig_path), width=800))

# Show some per-category comparisons
for fig_path in sorted(figures_dir.glob("*_comparison.png"))[:9]:
    print(f"\n{fig_path.name}:")
    display(Image(filename=str(fig_path), width=800))

In [ ]:
# Print metrics table
import json
with open("./results/tables/axis2_metrics.json") as f:
    metrics = json.load(f)

import sys
sys.path.insert(0, '.')
from interaction.verb_spatial_binding import print_metrics_table
print_metrics_table(metrics)

## Cell 7: Save Results to Drive

In [ ]:
import shutil
from pathlib import Path

# Copy results to Google Drive for persistence
drive_results = Path("/content/drive/MyDrive/VLA-affordance-results/axis2")
drive_results.mkdir(parents=True, exist_ok=True)

# Copy tables
shutil.copytree("./results/tables", str(drive_results / "tables"), dirs_exist_ok=True)

# Copy figures  
shutil.copytree("./results/figures/axis2", str(drive_results / "figures"), dirs_exist_ok=True)

# Copy cached attention maps if they exist
cache_dir = Path("./results/cached_features/flux_attention")
if cache_dir.exists() and any(cache_dir.iterdir()):
    shutil.copytree(str(cache_dir), str(drive_results / "attention_maps"), dirs_exist_ok=True)

print(f"✓ Results saved to: {drive_results}")